# Solución P03: Calidad y consulta diferida

[Enunciado](../talleres/P03.md) · [Índice](README.md)

Ejecutar de arriba abajo con el kernel del entorno `.venv` preparado en la [guía WSL](../docs/instalacion-wsl.md). Usa las tres muestras incluidas; no descarga los archivos completos. Las salidas propias se regeneran al repetir la ejecución.

In [ ]:
CODIGO = "P03"
from pathlib import Path
from types import SimpleNamespace
import sys, json
import pandas as pd
from IPython.display import display

# Funciona desde soluciones/, la raíz del repositorio o la carpeta base bigdata.
roots = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'big-data-postgraduate']
REPO = next((p for p in roots if (p / 'kit/pqrs_talleres.py').is_file()), None)
assert REPO is not None, 'Abrir Jupyter desde bigdata/big-data-postgraduate o soluciones/'
KIT = REPO / 'kit'
sys.path.insert(0, str(KIT))
import pqrs_talleres as lab
OUT = KIT / 'salidas/soluciones' / CODIGO
OUT.mkdir(parents=True, exist_ok=True)
a = SimpleNamespace(datos=None, completo=False, bloque=500, out=OUT, repeticiones=3)
print('Repositorio:', REPO, '\nSalidas:', OUT)


## Preparación y tarea 1. Calidad después de formatos
Este notebook prepara sus propios productos para poder ejecutarse independientemente de P02.

In [ ]:
lab.formatos(a)
control=lab.calidad(a)
assert control['filas']==3000 and control['exclusiones_automaticas']==0
assert control['claves_candidatas']==3000
assert control['mayores_90']==41 and control['ubicacion_peticionario_vacia']==3
display(pd.Series(control,name='resultado'))

## Tareas 2 y 3. Interpretación y control territorial
Las 3.000 claves candidatas distintas no demuestran ausencia de duplicados semánticos. Una proyección de 16 campos tampoco permite identificar duplicados exactos de los 38 campos originales. Las 41 edades mayores de 90 se revisan, no se eliminan por edad. Los tres registros sin ubicación del peticionario no prueban anonimato.

Para habilitar el control territorial, ejecutar previamente desde `kit/` el comando `python 00_datos.py --descargar divipola`, según la guía WSL. La celda siguiente informa expresamente cuando falta esa fuente.

In [ ]:
dim=KIT/'data/raw/divipola.csv'
if dim.exists():
    import duckdb
    dimension=pd.read_csv(dim,dtype=str)
    dimension['municipio']=dimension['Código Municipio'].str.zfill(5)
    with duckdb.connect() as con:
        con.register('dim',dimension[['municipio']].drop_duplicates())
        con.read_parquet(str(OUT/'pqrs.parquet')).create_view('q')
        territorial=con.sql("SELECT count(*) filas, count(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM dim WHERE municipio=q.pet_cod_mpio)) peticionarios_sin_coincidencia, count(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM dim WHERE municipio=q.afec_cod_mpio)) afectados_sin_coincidencia FROM q").df()
    display(territorial)
    territorial.to_csv(OUT/'control_territorial.csv',index=False)
else:
    print('Control territorial pendiente: falta DIVIPOLA; no se interpreta como cero incidencias.')

**Respuesta territorial:** conservar ambos campos y todos los registros. Investigar códigos vacíos, longitud, ceros iniciales y vigencia del catálogo antes de clasificar una falta de correspondencia como error.

## Tarea 4. Plan diferido y equivalencia SQL
Filtrar 2024 cambia el universo esperado a 2.000 filas. Conciliamos todos los grupos y no solo el total.

In [ ]:
import polars as pl
import duckdb
consulta=(pl.scan_parquet(OUT/'pqrs.parquet').filter(pl.col('anio')==2024).group_by(['anio','mes_num','afec_cod_depto']).agg(pl.len().alias('reportes')))
print(consulta.explain())
polars=consulta.collect().to_pandas().sort_values(['anio','mes_num','afec_cod_depto']).reset_index(drop=True)
with duckdb.connect() as con:
    sql=con.execute('SELECT anio,mes_num,afec_cod_depto,count(*) reportes FROM read_parquet(?) WHERE anio=2024 GROUP BY ALL ORDER BY ALL',[str(OUT/'pqrs.parquet')]).df()
pd.testing.assert_frame_equal(polars,sql,check_dtype=False)
assert int(sql.reportes.sum())==2000
display(sql)
sql.to_csv(OUT/'consulta_2024.csv',index=False)

## Tarea 5. Política de decisión y entrega
|Estado|Evidencia|Acción|
|---|---|---|
|Publicar con notas|Hashes y esquema válidos, conteos conciliados|Conservar trazabilidad y límites de cobertura|
|Revisar|Edad alta, vacíos territoriales, código sin correspondencia|Investigar con catálogo y fuente; conservar filas|
|Detener|Hash inesperado, esquema roto o pérdida de filas|Corregir y repetir validación antes de publicar|

**Impacto:** la política aplicada excluye cero filas; filtrar el año es una consulta explícita, no una limpieza del conjunto. La entrega incluye tabla de controles, consulta conciliada y, cuando esté disponible DIVIPOLA, control territorial. No se infieren prevalencia ni riesgo clínico de estos reportes.